# PB00 — Calcolo BACc per Soggetto (Subject-Specific Baseline)

Produce `data/interim/subject_bacc_pipelineB.csv` — letto da PB01 e PB03.

**Strategia**:
1. Carica bacc da W&B per i soggetti già runnati con DHSLP (EEG_13b) → risultati più accurati
2. Per i soggetti senza W&B (P074-P090, nuovi), calcola logistic regression su PSD features (~5-10s/sogg)

Questo garantisce consistenza interna: P000-P073 usano DHSLP (come da tesi), P074-P090 usano LR.
Per la correlazione con Blankertz il ranking relativo è ciò che conta, non il valore assoluto.

In [2]:
from pathlib import Path
import json

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT = project_root / 'data' / 'raw_csv' / 'training_set'
OUT_CSV   = project_root / 'data' / 'interim' / 'subject_bacc_pipelineB.csv'
CONFIGS   = project_root / 'configs' / 'label_schemes'

SFREQ = 256
N_CHAN = 61
N_SAMP = 384
CLUSTER_SCHEME = 'concr4'  # cambia in 'gram4' se preferisci

with open(CONFIGS / 'label2idx.json') as f:
    word2idx = json.load(f)
with open(CONFIGS / f'labelid2cluster_{CLUSTER_SCHEME}.json') as f:
    label2cluster = {int(k): v for k, v in json.load(f).items()}
N_CLASSES = len(set(label2cluster.values()))

print(f'Schema: {CLUSTER_SCHEME} | {N_CLASSES} classi | Chance: {1/N_CLASSES:.1%}')
print(f'Output: {OUT_CSV}')

Schema: concr4 | 4 classi | Chance: 25.0%
Output: /home/daniele_u/miralis-hypergraph-imagined-speech/data/interim/subject_bacc_pipelineB.csv


In [3]:
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
print('Import OK')

Import OK


In [4]:
# ============================================================
# STEP 1: carica bacc da W&B (EEG_13b runs già completate)
# ============================================================
import wandb

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'
WANDB_RUN_PREFIX = 'eeg13b'  # filtra run con questo prefisso nel nome

wandb_bacc = {}
try:
    api = wandb.Api()
    runs = api.runs(f'{WANDB_ENTITY}/{WANDB_PROJECT}',
                    filters={'display_name': {'$regex': f'^{WANDB_RUN_PREFIX}'}})
    for run in runs:
        # Ogni run subject-specific ha il soggetto nel nome: eeg13b_200e_P005_concr4
        name = run.name or ''
        import re
        m = re.search(r'P(\d+)', name)
        if m and run.state == 'finished':
            sid = int(m.group(1))
            bacc = run.summary.get('test_bacc') or run.summary.get('val_bacc')
            if bacc is not None and sid not in wandb_bacc:
                wandb_bacc[sid] = float(bacc)
    print(f'W&B: caricate {len(wandb_bacc)} run EEG_13b')
    if wandb_bacc:
        ids = sorted(wandb_bacc)
        print(f'  Soggetti: P{ids[0]:03d}–P{ids[-1]:03d}  |  bacc media: {sum(wandb_bacc.values())/len(wandb_bacc):.3f}')
except Exception as e:
    print(f'W&B non disponibile ({e}) — userò solo LR per tutti i soggetti')
    wandb_bacc = {}

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/daniele_u/.netrc.


W&B: caricate 90 run EEG_13b
  Soggetti: P000–P090  |  bacc media: 0.255


In [ ]:
# ============================================================
# STEP 2: calcola LR per soggetti senza risultati W&B
# ============================================================

all_subj = sorted(set(
    int(d.name.split('_')[0][1:]) for d in DATA_ROOT.iterdir() if d.is_dir()
))
print(f'Soggetti trovati sul disco: {len(all_subj)}')

missing_from_wandb = [s for s in all_subj if s not in wandb_bacc]
print(f'Già in W&B (DHSLP):    {len(wandb_bacc)}')
print(f'Da calcolare con LR:   {len(missing_from_wandb)}  → {[f"P{s:03d}" for s in missing_from_wandb]}')

# Salta soggetti già calcolati con LR nelle run precedenti (resume)
if OUT_CSV.exists():
    _prev = pd.read_csv(OUT_CSV)
    done_lr = set(_prev['subj_id'].astype(int).tolist())
    print(f'Già calcolati (LR precedente): {len(done_lr & set(missing_from_wandb))}')
else:
    done_lr = set()
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

results = []
clf = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=0.1, max_iter=500, solver='lbfgs'))
])

for sid in tqdm(missing_from_wandb, desc='LR per soggetti nuovi'):
    if sid in done_lr:
        continue

    X_raw, y = load_subject_img(sid)
    if len(X_raw) < N_CLASSES * 5:
        results.append({'subj_id': sid, 'bacc': np.nan, 'n_trials': len(X_raw), 'source': 'lr_insufficient'})
        continue

    X = psd_features(X_raw)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    baccs = []
    for tr, te in cv.split(X, y):
        clf.fit(X[tr], y[tr])
        baccs.append(balanced_accuracy_score(y[te], clf.predict(X[te])))

    results.append({'subj_id': sid, 'bacc': float(np.mean(baccs)), 'n_trials': len(X_raw), 'source': 'lr_psd'})

    # Salva incrementalmente
    df_new = pd.DataFrame(results)
    if OUT_CSV.exists():
        df_new = pd.concat([pd.read_csv(OUT_CSV), df_new], ignore_index=True)
    df_new.to_csv(OUT_CSV, index=False)
    results = []

# ============================================================
# Merge W&B + LR → CSV finale
# ============================================================
rows_wandb = [{'subj_id': s, 'bacc': b, 'n_trials': None, 'source': 'dhslp_wandb'}
              for s, b in wandb_bacc.items()]

if OUT_CSV.exists():
    df_existing = pd.read_csv(OUT_CSV)
    df_lr_only  = df_existing[~df_existing['subj_id'].isin(wandb_bacc)]
else:
    df_lr_only = pd.DataFrame(columns=['subj_id', 'bacc', 'n_trials', 'source'])

df_final = pd.concat([pd.DataFrame(rows_wandb), df_lr_only], ignore_index=True)
df_final = df_final.sort_values('subj_id').reset_index(drop=True)
df_final.to_csv(OUT_CSV, index=False)

print(f'\nCSV finale: {len(df_final)} soggetti')
print(df_final['source'].value_counts().to_string())
print(f'\nSalvato in: {OUT_CSV}')

Soggetti trovati sul disco: 91
Già in W&B (DHSLP):    90
Da calcolare con LR:   1  → ['P073']


TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

In [ ]:
# ============================================================
# LOOP PRINCIPALE — bacc per soggetto con 5-fold CV
# ============================================================

all_subj = sorted(set(
    int(d.name.split('_')[0][1:]) for d in DATA_ROOT.iterdir() if d.is_dir()
))
print(f'Soggetti trovati: {len(all_subj)}')

# Salta soggetti già calcolati (resume)
if OUT_CSV.exists():
    done = pd.read_csv(OUT_CSV)['subj_id'].tolist()
    print(f'Già calcolati: {len(done)} — riprendo da dove ero')
else:
    done = []
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

results = []
clf = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=0.1, max_iter=500, solver='lbfgs', multi_class='multinomial'))
])

for sid in tqdm(all_subj, desc='Soggetti'):
    if sid in done:
        continue

    X_raw, y = load_subject_img(sid)
    if len(X_raw) < N_CLASSES * 5:  # troppo pochi trial
        results.append({'subj_id': sid, 'bacc': np.nan, 'n_trials': len(X_raw)})
        continue

    X = psd_features(X_raw)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    baccs = []
    for tr, te in cv.split(X, y):
        clf.fit(X[tr], y[tr])
        baccs.append(balanced_accuracy_score(y[te], clf.predict(X[te])))

    bacc = float(np.mean(baccs))
    results.append({'subj_id': sid, 'bacc': bacc, 'n_trials': len(X_raw)})

    # Salva incrementalmente
    df_new = pd.DataFrame(results)
    if OUT_CSV.exists():
        df_new = pd.concat([pd.read_csv(OUT_CSV), df_new], ignore_index=True)
    df_new.to_csv(OUT_CSV, index=False)
    results = []  # reset buffer

print(f'\nDone. Risultati in: {OUT_CSV}')

In [ ]:
# ============================================================
# RIEPILOGO
# ============================================================
import matplotlib.pyplot as plt

df = pd.read_csv(OUT_CSV).sort_values('bacc', ascending=False)
chance = 1 / N_CLASSES

print(f'Soggetti totali: {len(df)}')
print(f'bacc media: {df["bacc"].mean():.3f}  (chance={chance:.3f})')
print(f'BCI-illiterate (bacc ≤ chance+5%): {(df["bacc"] <= chance + 0.05).sum()}')
print(f'BCI-literate   (bacc > 35%):       {(df["bacc"] > 0.35).sum()}')

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['steelblue' if b > 0.30 else 'tomato' for b in df['bacc']]
ax.bar(range(len(df)), df['bacc'], color=colors, alpha=0.8)
ax.axhline(chance, color='k', linestyle='--', label=f'Chance ({chance:.2f})')
ax.set_xlabel('Soggetto')
ax.set_ylabel('bacc (5-fold CV)')
ax.set_title(f'Performance per soggetto — {CLUSTER_SCHEME}')
ax.legend()
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB00_subject_bacc_distribution.png', dpi=150)
plt.show()